# HDF5 Dataset Inspection

检查 `prostate_classification/patients_dataset_v1.0.h5` 的内容结构。

In [ ]:
import os
import h5py
import numpy as np
from collections import Counter

# 从 data/ 目录上一级到 PCa-HSD-LSDT/，再进 prostate_classification/
H5_PATH = os.path.join(os.path.dirname(os.getcwd()), "prostate_classification", "patients_dataset_v1.0.h5")
print(f"Looking for HDF5 file at: {H5_PATH}")
print(f"Exists: {os.path.exists(H5_PATH)}")

if not os.path.exists(H5_PATH):
    raise FileNotFoundError(
        f"HDF5 file not found!\n"
        f"Expected path: {H5_PATH}\n"
        f"Please place your patients_dataset_v1.0.h5 file in: "
        f"{os.path.join(os.path.dirname(os.getcwd()), 'prostate_classification')}"
    )

In [ ]:
with h5py.File(H5_PATH, "r") as f:
    patient_ids = list(f.keys())
    
print(f"Total patients: {len(patient_ids)}")
print(f"\nFirst 10 IDs: {patient_ids[:10]}")

In [ ]:
# 检查每个 patient group 下的内容结构
with h5py.File(H5_PATH, "r") as f:
    g = f[patient_ids[0]]
    print(f"Keys in patient group '{patient_ids[0]}':")
    for k in g.keys():
        v = g[k]
        if hasattr(v, 'shape'):
            print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"  {k}: {v[()]}")

In [ ]:
# 标签分布统计
labels = []
with h5py.File(H5_PATH, "r") as f:
    for pid in patient_ids:
        label = int(f[pid]["label"][()])
        labels.append(label)

print("Label distribution:")
counter = Counter(labels)
for label in sorted(counter.keys()):
    print(f"  Class {label}: {counter[label]} samples ({100*counter[label]/len(labels):.1f}%)")

In [ ]:
# 检查是否有 mask 字段
has_mask = 0
no_mask = 0
with h5py.File(H5_PATH, "r") as f:
    for pid in patient_ids:
        if "mask_res" in f[pid]:
            has_mask += 1
        else:
            no_mask += 1
print(f"Patients with mask_res:   {has_mask}")
print(f"Patients without mask_res: {no_mask}")

In [ ]:
# 检查数据范围（以第一个 patient 为例）
with h5py.File(H5_PATH, "r") as f:
    g = f[patient_ids[0]]
    print(f"Patient {patient_ids[0]} modality value ranges:")
    for mod in ["ADC", "DWI", "T2", "ADC_res", "DWI_res", "T2_res"]:
        if mod in g:
            data = g[mod][:]
            print(f"  {mod}: min={data.min():.4f}, max={data.max():.4f}, mean={data.mean():.4f}, std={data.std():.4f}")
    if "mask_res" in g:
        mask = g["mask_res"][:]
        print(f"  mask_res: unique values={np.unique(mask)}")

In [ ]:
# 检查所有 patient 的切片数（S 维度）是否一致
slice_counts = []
with h5py.File(H5_PATH, "r") as f:
    for pid in patient_ids:
        s = f[pid]["ADC"].shape[0]
        slice_counts.append(s)

arr = np.array(slice_counts)
print(f"Slice counts (ADC): min={arr.min()}, max={arr.max()}, mean={arr.mean():.1f}")
if arr.min() != arr.max():
    print("  [WARNING] Slice counts are NOT uniform across patients!")
    print(f"  Unique values: {sorted(set(slice_counts))}")
else:
    print(f"  All patients have {arr[0]} slices.")

In [ ]:
# 检查是否有属性信息（如 early_stage_cancer）
with h5py.File(H5_PATH, "r") as f:
    g = f[patient_ids[0]]
    attrs = dict(g.attrs)
    print(f"Attributes for patient {patient_ids[0]}:")
    if attrs:
        for k, v in attrs.items():
            print(f"  {k}: {v}")
    else:
        print("  (none)")

In [ ]:
print("=== Inspection Complete ===")
print(f"Total patients: {len(patient_ids)}")
print(f"Number of classes: {len(set(labels))}")
print(f"HDF5 file size: ", end="")
import os
size = os.path.getsize(H5_PATH)
if size < 1024**2:
    print(f"{size/1024:.1f} KB")
elif size < 1024**3:
    print(f"{size/1024**2:.1f} MB")
else:
    print(f"{size/1024**3:.1f} GB")